## Hi Nicole, this is a to-do on how to load lightcurve data and phase fold using 5 high clock quality examples

In [ ]:
#import packages
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import median_filter
from scipy.interpolate import CubicSpline

import lightkurve as lk
from astropy import units as u
from astropy.table import Table
from astropy.io import ascii
from astropy.time import Time, TimeDelta

import time
import sqlite3 as db
import os
import sys

### define global variables

In [ ]:
f_avoid = 3.5 / 372.5 #magic number
lc_exptime = (29.4) / (60 * 24) #days, see Kepler Data Processing Handbook, Section 3.1
sc_exptime = (58.8) / (60 * 60 * 24) #days, see Kepler Data Processing Handbook, Section 3.1
fiducial_bjd = 0 #magic number, placeholder 

## Functions!

### Check if lightcurve is ordered and reorder

In [ ]:
def check_inputs(xs):
    """
    ## Inputs:
    `xs`: list of lc time values (numpy array)

    ## Outputs:
    `bool`: `True` if the array is sorted, `False` otherwise
    """
    for i in range(len(xs) - 1):
        if xs[i] > xs[i + 1]:
            return False
    return True

def reorder_inputs(xs, ys):
    """
    ## Inputs:
    `xs`: lc time values (numpy array)  
    `ys`: lc flux values (numpy array)

    ## Outputs:
    A tuple `(xs_sorted, ys_sorted)` where:
    - `xs_sorted`: `xs` sorted in ascending order
    - `ys_sorted`: corresponding `ys` values reordered to match `xs_sorted`

    ## Bugs:
    - Assumes `xs` and `ys` are NumPy arrays
    - Raises a ValueError if `xs` and `ys` have different lengths
    """
    if len(xs) != len(ys):
        raise ValueError("reorder_inputs(): `xs` and `ys` must be the same length")
    i = np.argsort(xs)
    return xs[i], ys[i]

### Get the lightcurve from kic id 
and return lc, 1/total observation time, sampling time, and exposure time

In [ ]:
def get_kepler_data(kic_id, exptime='long'):
    
    start = time.time()
    """
    ## Inputs:
    `kic_id`: Kepler ID (str)
    `exptime`: default exposure time, 'long'

    ## Outputs:
    Returns a 4-tuple:
    - `lc`:  Lightkurve lc object (or nan if failed)
    - `delta_f`: frequency resolution, 1 / total observation time
    - `sampling_time`: median time between observations (in days)
    - `exptime`:  exposure time in days (from global `lc_exptime` or `sc_exptime`)

    ## Bugs:
    - Depends on global vals: `lc_exptime`, `sc_exptime` 
    - Fails silently when no data found
    - Rejects light curves where any `dt < 0.9 * median(dt)` — may be too strict
    - Uses magic thresholds for time sampling
    """
    print("starting to download data for", kic_id)

    try:
        search_result = lk.search_lightcurve(kic_id, mission = 'Kepler', exptime=exptime)
    
        if len(search_result) < 1:
            msg = f"get_kepler_data(): no results for {kic_id} at this cadence"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)

            return np.nan, np.nan, np.nan, np.nan

   
        lc_collection = search_result.download_all()
        lc = lc_collection.stitch()



        if check_inputs(lc.time.value) is False:
            time_val, __ = reorder_inputs(lc.time.value, lc.flux.value)
        else:
            time_val, __ = lc.time.value, lc.flux.value

        if not np.all(np.diff(time_val) > 0):
            print("nana.star(): times not in order")
            update_error_message(kic_id, 'Kepler_long', "Times not in order")
    
        
        delta_f = (1/(time_val[-1] - time_val[0]))
        
        sampling_time= np.median(np.diff(time_val))

        if not np.all(np.diff(time_val) > 0.90 * sampling_time): #magic
            print("nana.star(): some time intervals out of spec")
            print("nana.star(): median dt = ", sampling_time)
            update_error_message(kic_id, 'Kepler_long', "Some time intervals out of spec")
        
        exptime = np.nan
        if sampling_time > 0.9 * lc_exptime:  #magic number?
            exptime = lc_exptime
            print(f"{kic_id} is long cadence")
        if sampling_time < 1.1 * sc_exptime: #also magic number?
            exptime = sc_exptime
            print(f"{kic_id} is short cadence")
        if np.isnan(exptime):
            msg = f"{kic_id}: get_kepler_data(): no consistent exptime"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)
            return np.nan, np.nan, np.nan, np.nan
        print("get_kepler_data() took", time.time() - start, "seconds")

    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): final processing failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan

    return lc, delta_f, sampling_time, exptime

### masking the lightcurve to get rid of bad data

In [ ]:
def mask_vals(lc):
    """
    Remove mask values from a Lightkurve LightCurve object and return valid time, flux, and weight arrays.

    ## Inputs:
    `lc`: Lightkurve LightCurve object:
    - `

    ## Outputs:
    3 NumPy arrays:
    - `t_fit`: time values with valid (finite) data
    - `flux_fit`: corresponding flux values
    - `weight_fit`: corresponding weights (1 / sigma^2)

    ## Bugs:
    - not obvious that we need this function
    """
    #replaces masked values with NaN
    #print(type(lc.flux)) --> confirmed that lc.flux is a MaskedArray

    try: 
        t_clean = np.ma.filled(lc.time.value, np.nan)
        flux_clean = np.ma.filled(lc.flux.value, np.nan)
        sigma_clean = np.ma.filled(lc.flux_err.value, np.nan)

        #gets rid of NaNs, creates mask with only finite/valid values
        mask = np.isfinite(t_clean) & np.isfinite(flux_clean) & np.isfinite(sigma_clean)
        t_fit = t_clean[mask]
        flux_fit = flux_clean[mask]
        sigma_fit = sigma_clean[mask]
        weight_fit = 1 / sigma_fit**2
    
    except Exception as e:
        print(f"Exception in mask_vals(): {str(e)}")
        #update_error_message(kic_id, 'Kepler_long', str(e))
        return np.nan, np.nan, np.nan

    return t_fit, flux_fit, weight_fit

### functions for refinement

In [ ]:
def integral_chi_squared(om, ts, ys, ws, T, M = 1):
    """
    ## Inputs:
    `om`: angular frequency (in inverse days)  
    `ts`: numpy array of observation times (in days)  
    `ys`: numpy array of observed flux values  
    `ws`: numpy array of weights (same length as `ts` and `ys`)  
    `T`: exposure time (in days)
    `M`: number of harmonics to include in the model (default is 1)

    ## Outputs:
    Weighted chi-squared value computed using the integral design matrix model

    ## Bugs:
    - Assumes all inputs are NumPy arrays of compatible shapes
    - Assumes uniform exposure time `T` for all observations
    - Numerically unstable when `om * T` is small (from `integral_design_matrix`)
    """
    A = integral_design_matrix(ts, om, T, M = M)
    return np.sum(ws * (ys - (A @ weighted_least_squares(A, ys, ws)))**2)


def find_min_and_refine(xs, ys):
    """
    ## Inputs:
    `xs`: numpy array of x values  
    `ys`: numpy array of y values 

    ## Outputs:
    Tuple `(refined_x, refined_y)` where:
    - `refined_x`: x-position of the refined minimum
    - `refined_y`: y-value at the refined minimum

    ## Bugs:
    - Code header needs to say what this code does
    - Assumes `xs` and `ys` are NumPy arrays of equal length
    - Assumes `xs` is ordered
    - Raises `ValueError` if no local minima are found
    - Raises `IndexError` if minimum is at the edge (index 0 or len-1)
    """
    #ys = np.asarray(list(ys))
    
    try: 
        indxs, _ = find_peaks(-ys)

        if len(indxs) == 0:
            msg = "find_min_and_refine(): no local minima found"
            print
            return np.nan, np.nan
            #raise ValueError("find_min_and_refine(): no local minima found")
        
        min_index = indxs[np.argsort(ys[indxs])[:1]]
        if min_index < 1 or min_index > len(xs) - 2:
            msg = "find_min_and_refine(): minimum too close to edge to refine"
            print(msg)
            return np.nan, np.nan
        
        refined_x, refined_y, _ = refine_peaks(xs, ys, min_index)

    except Exception as e:
        print(f"Exception in find_min_and_refine(): {str(e)}")
        return np.nan, np.nan
    
    return refined_x[0], refined_y[0]

def weighted_least_squares(A, b, weights):
    """
    ## Inputs:
    `A`: NxM design matrix (NumPy array)  
    `b`: N-length observation vector (NumPy array)  
    `weights`: N-length vector of weights (NumPy array), applied per row

    ## Outputs:
    N-length fitted model values `A @ x`, where `x` solves the weighted least squares problem

    ## Bugs:
    - Assumes all inputs are NumPy arrays of compatible shape
    """

    ATA = A.T @ (A * weights[:, np.newaxis])
    ATb = A.T @ (b * weights)
    return np.linalg.solve(ATA, ATb)

def integral_design_matrix(ts, om, T, M = 1):
    """
    ## Inputs:
    `ts`: list of N times (days)
    `om`: angular frequency (inverse days)
    `T`: exposure time (days)
    `M`: number of harmonics (default is 1)

    ## Outputs:
    `X`: Nx3 design matrix

    ## Bugs:
    - Assumes all data points have the same exposure time `T`
    - Not numerically stable when `om * T` is small
    """
    X =  np.vstack([
        np.ones_like(ts),
        (np.sin(om * (ts + T / 2)) - np.sin(om * (ts - T / 2))) / (om * T),
        (-np.cos(om * (ts + T / 2)) + np.cos(om * (ts - T / 2))) / (om * T)
    ]).T
    for m in range(2, M + 1):
        X = np.hstack((X, np.vstack([
            (np.sin(m * om * (ts + T / 2)) - np.sin(m * om * (ts - T / 2))) / (m * om * T),
            (-np.cos(m * om * (ts + T / 2)) + np.cos(m * om * (ts - T / 2))) / (m * om * T)
        ]).T))
    return X

def design_matrix(xlist):
    """
    ## Inputs:
    `xlist`: numpy array of length 3 for the three frequency points (middle and two neighbors)

    ## Outputs:
    A 3x3 design matrix:
    - Column 1: constant term (1s)
    - Column 2: linear term (`xlist`)
    - Column 3: quadratic term with 0.5 factor (`0.5 * xlist**2`)

    ## Bugs:
    - Assumes `xlist` has length 3 
    - Assumes `xlist` is ordered

    ## Notes:
    - Includes a 0.5 factor that Hogg likes in the quadratic term 
    """
    return (np.vstack((xlist**0, xlist**1, 0.5 * xlist**2))).T

def fit_parabola(xs, ys, index):
    
    """
    ## Inputs:
    `xs`: numpy array of frequency values
    `ys`: numpy array of power values 
    `index`: integer index of the central point to fit around

    ## Outputs:
    Tuple `(b, m, q)` representing coefficients of the quadratic:  

    ## Bugs:
    - `xs` and `ys` must be numpy arrays
    - Index must not be 0 or `len(xs) - 1`; otherwise will be out of bounds
    """
    #index = int(index)
    if index < 1 or index > len(xs) - 2:
        raise IndexError("fit_parabola(): index must be between 1 and len(xs) - 2")
    return np.linalg.solve(design_matrix(xs[index-1:index+2]), ys[index-1:index+2])

def refine_peak(xs, ys, index):
    """
    ## Inputs:
    `xs`: numpy array of frequency values 
    `ys`: numpy array of power values
    `index`: integer index of peak to refine

    ## Outputs:
    3-tuple `(x_peak, y_peak, q)` where:
    - `x_peak`: refined x-position of the peak 
    - `y_peak`: refined y-position
    - `q`: second derivative

    ## Bugs:
    - Must be synchronized with the design matrix (uses same quadratic form)
    """
    b, m, q = fit_parabola(xs, ys, index)
    x_peak = -m / q
    if x_peak < xs[0] or x_peak > xs[-1]:
        return np.nan, np.nan, np.nan
    y_peak = 0.5 * q * x_peak**2 + m * x_peak + b
    return x_peak, y_peak, q

def refine_peaks(xs, ys, indices):
    """
    ## Inputs:
    `xs`: numpy array of frequency values
    `ys`: numpy array of power values
    `indices`: numpy array of frequency peak indices

    ## Outputs:
    Three NumPy arrays:
    - `xs_refined`: refined x positions of peaks
    - `ys_refined`: refined y positions of peaks
    - `second_derivatives`: curvature values (second derivative q for each peak)

    ## Bugs:
    - Code header needs to say what this code does
    - Assumes all `indices` are valid (i.e., between 1 and len(xs) - 2) 
    - Assumes `xs` and `ys` are numpy arrays and ordered
    """
    try: 
        n = len(indices)
        xs_refined = np.full(n, np.nan)
        ys_refined = np.full(n, np.nan)
        second_derivatives = np.full(n, np.nan)
        
        for j, i in enumerate(indices):
            if np.isnan(i):
                continue
            else:
                result = refine_peak(xs, ys, i)
                x_r, y_r, q_r = result
                xs_refined[j] = x_r
                ys_refined[j] = y_r
                second_derivatives[j] = q_r
    except Exception as e:
        print(f"Exception in refine_peaks(): {str(e)}")
        return np.nan, np.nan, np.nans
            
    return np.array(xs_refined), np.array(ys_refined), np.array(second_derivatives)

In [ ]:
#choosing the number of coefficients in the fourier series 
def choose_M(sample_time, freq):
    nyquist = 1.0 / (2 * sampling_time)    
    M = np.round(0.5 * nyquist / freq).astype(int)
    if M > 64:
        M = 64
    assert M != 0, f"find_M() returned 0 for freq={freq}"
    return M

### created your folded lightcurve!!!

In [ ]:
def phase_fold(flux_fit, refined_frequency, coeffs, M):


    
        fig = plt.figure()
        theta = np.linspace(0, 4 * np.pi, 1000)
        yplot0 = np.zeros_like(theta)
        
        for m in range(1, M + 1):
            a = coeffs[2*m - 1]
            b = coeffs[2*m]
            yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
        yplot0 += coeffs[0]

        phase = (om * t_fit) % (4 * np.pi)

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(phase, flux_fit, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k', rasterized=True)
        ax.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
        ax.set_xlabel('Phase')
        ax.set_ylabel('Flux')
        ax.set_title(rf'freq = {refined_frequency:.04f} $day^{{-1}}$, M = {M}')
        #
        plt.show()
        plt.close(fig)
            

## example run

In [ ]:
kicID_sample, frequency = "KIC006780873", 14.187641893784763 #this frequency needs to be refined but the following will refine

# get the lightcurve for this kicid
lc, delta_f, sampling_time, exptime = get_kepler_data(kicID_sample)
t_fit, flux_fit, weight_fit = mask_vals(lc)

# choose M
M_val = choose_M(sampling_time, frequency)

# get chi2 values at 3 frequencies near the frequency
epsilon = 0.5 * delta_f
freqs = np.array([-epsilon * 3, 0, epsilon * 3]) + frequency
chi2s = np.zeros_like(freqs)
for i in range(len(freqs)):
    chi2s[i] = integral_chi_squared(2 * np.pi * freqs[i], t_fit, flux_fit, weight_fit, exptime, M=M_val)

assert np.argmin(chi2s) == 1, (
f"Central point is not the minimum, "
f"chi2s={chi2s}"
)

refined_freq, refined_chi2 = find_min_and_refine(freqs, chi2s)

# Perform fourier series fit to degree M
om = refined_freq * 2 * np.pi
A = integral_design_matrix(t_fit, om, exptime, M = M_val)
coeffs = weighted_least_squares(A, flux_fit, weight_fit)

#you have all your values to phase fold! you should probably include the kic id here
phase_fold(flux_fit, refined_freq, coeffs, M_val)

### YAY
Try testing on these kic ids with the corresponding

In [ ]:
data = np.load('kics_and_freqs_for_nicole.npy')
print(data)